# Fine-tune Cross-Encoder CV-JD v0.5

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **MSE regression loss** trên dataset v0.4 đã relabel bằng LLM.

| | |
|---|---|
| **Dataset** | v0.4 — 4900 train / 1050 validation / 1050 test (7000 pairs, 4188 relabeled by LLM) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` regression (label = score / 100) |
| **Evaluator** | Spearman correlation + LabelAcc |
| **max_length** | 512 tokens |
| **Branch** | `develop` |

**Mục tiêu**: vượt LabelAcc 60.76% của v0.2 nhờ data quality cao hơn (relabeled 4188/7000 pairs bởi LLM với rubric v0.2).

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout develop
!git pull origin develop

In [ ]:
!pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.4/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Debug run — sanity check (1 epoch, 40 samples)

In [ ]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.4/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20

## Full training — 10 epochs, save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.5", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.4/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.5 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.5_report.json \
    --epochs 10 \
    --batch-size 16

In [ ]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.5_report.json").read_text(encoding="utf-8")
)
print(f"Base model : {report['base_model']}")
print(f"Loss       : {report['loss']}")
print()
print(json.dumps(report["metrics"], indent=2))

In [ ]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.5_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.5_report.json",
)
print(f"Saved to {reports_dir}")